<a href="https://colab.research.google.com/github/softnextgr/bruteforceai/blob/main/captcha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNIST CAPTCHA

Αυτό το starter notebook εστιάζει στην επίλυση του προβλήματος MNIST CAPTCHA. Στόχος είναι να σχεδιάσουμε ένα σύστημα που αναγνωρίζει τα ψηφία και τον τελεστή από μία εικόνα τύπου "AB ± CD", ώστε να υπολογίζει το τελικό αποτέλεσμα. Το notebook παρέχει τα ελάχιστα βήματα: εισαγωγές βιβλιοθηκών, λήψη του MNIST dataset και έναν απλό ταξινομητή, ώστε να συνεχίσετε με δικές σας υλοποιήσεις.

**Αυτό το Notebook είναι Read-Only** - οπότε πρέπει να αποθηκεύσετε ένα αντίγραφο στο δικό σας Google Drive.


## Εισαγωγές βιβλιοθηκών

Χρειαζόμαστε την NumPy για διαχείριση πινάκων και την PyTorch για τη δημιουργία και εκπαίδευση νευρωνικών δικτύων. Θα χρησιμοποιήσουμε επίσης το torchvision για να κατεβάσουμε το MNIST dataset μαζί με βασικούς μετασχηματισμούς.


In [ ]:
import numpy as np
import torch
from torch import nn
from torchvision import datasets, transforms
from gdown import download
from PIL import Image

## Λήψη των αρχείων

In [ ]:
download(id='18IZn5DroVvTkGJEKn5w15RuT61ET9HP0', output='public-clean.png', quiet=False)
download(id='1y3xX9VrM7EtYf1W-3vr-PETE19FnTKHR', output='public-clean.txt', quiet=False)

download(id='1cvDlXQrkLvR_tQBydyms-dt-VSjB7RlG', output='public-noisy.png', quiet=False)
download(id='1V-e76Q8Op3FWFEqb42bvJKllM5LfCdVS', output='public-noisy.txt', quiet=False)

download(id='1JfbYOpBHNrlz-fqgOtLZ8QDidxznug7U', output='private-clean.png', quiet=False)
download(id='1WtozDPV0FjmPthKBBiqBPfd-lhCMxfzk', output='private-noisy.png', quiet=False)

Downloading...
From: https://drive.google.com/uc?id=18IZn5DroVvTkGJEKn5w15RuT61ET9HP0
To: /content/public-clean.png
100%|██████████| 1.92M/1.92M [00:00<00:00, 108MB/s]
Downloading...
From: https://drive.google.com/uc?id=1y3xX9VrM7EtYf1W-3vr-PETE19FnTKHR
To: /content/public-clean.txt
100%|██████████| 8.47k/8.47k [00:00<00:00, 10.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1cvDlXQrkLvR_tQBydyms-dt-VSjB7RlG
To: /content/public-noisy.png
100%|██████████| 2.89M/2.89M [00:00<00:00, 173MB/s]
Downloading...
From: https://drive.google.com/uc?id=1V-e76Q8Op3FWFEqb42bvJKllM5LfCdVS
To: /content/public-noisy.txt
100%|██████████| 8.49k/8.49k [00:00<00:00, 12.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1JfbYOpBHNrlz-fqgOtLZ8QDidxznug7U
To: /content/private-clean.png
100%|██████████| 1.92M/1.92M [00:00<00:00, 107MB/s]
Downloading...
From: https://drive.google.com/uc?id=1WtozDPV0FjmPthKBBiqBPfd-lhCMxfzk
To: /content/private-noisy.png
100%|██████████| 2.89M/2.89M [00:00<00:0

'private-noisy.png'

In [ ]:
import os
def showImage(tensor):
    transform = transforms.ToPILImage()
    return transform(1-tensor)
def readExamples(file_name):
    transform = transforms.ToTensor()
    image = Image.open(file_name + '.png').convert('L')
    tensor = 1-transform(image)
    X = tensor.reshape(-1,1,28,140)
    if os.path.exists(file_name + '.txt'):
        with open(file_name + '.txt') as f:
            lines = f.readlines()
    else:
        lines = ['']*X.shape[0]
    return X, lines

examples, results = readExamples('public-clean') # or 'public-noisy' or 'private-clean' or 'private-noisy'
print(results[0])
showImage(examples[0])

71



## Λήψη του MNIST dataset

Θα κατεβάσουμε το κλασικό MNIST dataset μέσω του torchvision και θα το μετατρέψουμε σε tensors ώστε να χρησιμοποιηθεί αργότερα στην εκπαίδευση.


In [ ]:
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.38MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 158kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.50MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.0MB/s]

Train samples: 60000
Test samples: 10000


In [ ]:
image, label = train_dataset[0]
print(label)
showImage(image)

5


# Η Βασική Ιδέα

Για να αυτοματοποιήσουμε την επίλυση των MNIST-CAPTCHA, πρέπει να εκπαιδεύσουμε ένα νευρωνικό που θα μπορεί να αποκωδικοποιήσει το μαθηματικό πρόβλημα από τα ψηφία του MNIST.

Το πρώτο βήμα είναι να εκπαιδεύσουμε ένα νευρωνικό δίκτυο που μπορεί να αναγνωρίσει να ψηφία ένα ένα.

Το επόμενο βήμα είναι να ξεχωρίσουμε τα ψηφία από το Captcha πρόβλημα, να χρησιμοποιήσουμε το εκπαιδευμένο νευρωνικό μας δίκτυο, και μετά να υπολογίσουμε την απάντηση, έχοντας βρεί τους σωστούς αριθμούς.

## Απλό νευρωνικό δίκτυο

Ως ελάχιστη εκκίνηση, ορίζουμε ένα μοντέλο με ένα μόνο γραμμικό επίπεδο (layer) `nn.Linear(28*28, 10)` το οποίο ισοπεδώνει τα pixels και παράγει logits για τις 10 κλάσεις του MNIST.


In [ ]:

class SimpleMNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(28 * 28, 10)


    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = SimpleMNISTModel()
print(model)


SimpleMNISTModel(
  (fc): Linear(in_features=784, out_features=10, bias=True)
)


## Υλοποίηση εκπαίδευσης

Στο επόμενο βήμα θα χρειαστεί να γράψετε τη δική σας συνάρτηση εκπαίδευσης ώστε να προσαρμόσετε το μοντέλο στα δεδομένα και να το επεκτείνετε για τη λύση του MNIST CAPTCHA.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


# TODO: Γράψτε τον κώδικα εκπαίδευσης του μοντέλου

SimpleMNISTModel(
  (fc): Linear(in_features=784, out_features=10, bias=True)
)

In [ ]:
# Με βάση το εκπαιδευμένο μοντέλο υλοποιήστε μια συνάρτηση που υπολογίζει την πράξη στο δοσμένο παράδειγμα
def predict(model, x):
    #TODO
    return "37+34=71" #placeholder

## Αποθήκευση Απαντήσεων

In [ ]:
import json
answers = {}
for level in ['public-clean', 'public-noisy', 'private-clean', 'private-noisy']:
    examples, _ = readExamples(level)
    answers[level] = '\n'.join( [predict(model, example) for example in examples] )
with open('answers.json', 'w') as f:
    json.dump(answers, f)

In [ ]:
from google.colab import files
files.download('answers.json')